### Notebook for exploring Self-Conditioned Knots for Probabilistic LiDAR Hazrad Fields

In [10]:
from vedo import *
settings.default_backend = "k3d"
import numpy as np
from scipy.spatial.transform import Rotation as R
import copy

from matplotlib import pyplot as p

%load_ext autoreload
%autoreload 2
%autosave 180

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Autosaving every 180 seconds


## Generate synthetic LiDAR data

In [25]:
from synthetic_data_generation_utils import (
    LidarProjectionModel,
    Scene,
    generate_point_cloud,
)

# 1. LiDAR Intrinsics (e.g. 32-channel, 360-degree rotation)
sensor = LidarProjectionModel(
    elevation_min=-25.0,
    elevation_max=25.0,
    num_detectors=32,
    num_azimuth=360,
    max_range=30.0,
)

# 2. Build 10x10m Enclosed Scene
scene = Scene()
wall_density = 100.0  # Solid boundaries (near-surface returns)

# --- Floor & 4 Walls (10m x 10m x 3m interior) ---
scene.add_box(center=[0.0, 0.0, -0.1], size=[10.4, 10.4, 0.2], density=wall_density, name="floor")
scene.add_box(center=[0.0, 5.1, 1.5],  size=[10.4, 0.2, 3.0],  density=wall_density, name="wall_north")
scene.add_box(center=[0.0, -5.1, 1.5], size=[10.4, 0.2, 3.0],  density=wall_density, name="wall_south")
scene.add_box(center=[5.1, 0.0, 1.5],  size=[0.2, 10.4, 3.0],  density=wall_density, name="wall_east")
scene.add_box(center=[-5.1, 0.0, 1.5], size=[0.2, 10.4, 3.0],  density=wall_density, name="wall_west")

# --- 4 Cylinders (Pillars) with density = 1.0 ---
cyl_density = 1.0
scene.add_cylinder(center=[2.5, 2.5, 1.5],   radius=0.4, height=3.0, density=cyl_density, name="pillar_NE")
scene.add_cylinder(center=[-2.5, 2.5, 1.5],  radius=0.4, height=3.0, density=cyl_density, name="pillar_NW")
scene.add_cylinder(center=[2.5, -2.5, 1.5],  radius=0.4, height=3.0, density=cyl_density, name="pillar_SE")
scene.add_cylinder(center=[-2.5, -2.5, 1.5], radius=0.4, height=3.0, density=cyl_density, name="pillar_SW")

# --- Low-Density Spheres (Porous foliage / clutter with deep volumetric penetration) ---
scene.add_sphere(center=[2.0, 0.0, 1.0],   radius=0.8,  density=0.15, name="sphere_1")
scene.add_sphere(center=[-2.0, 0.5, 1.2],  radius=0.9,  density=0.10, name="sphere_2")
scene.add_sphere(center=[0.0, 3.0, 1.5],   radius=1.1,  density=0.08, name="sphere_3")
scene.add_sphere(center=[0.0, -2.8, 1.0],  radius=0.7,  density=0.20, name="sphere_4")
scene.add_sphere(center=[3.2, -1.8, 1.3],  radius=1.0,  density=0.12, name="sphere_5")
scene.add_sphere(center=[-3.0, -2.0, 1.4], radius=0.85, density=0.18, name="sphere_6")

# 3. Generate Point Cloud with the sensor positioned at the center of the room [0, 0, 1.5]
sensor_pos = [0.0, 0.0, 1.5]
point_cloud = generate_point_cloud(
    sensor=sensor,
    scene=scene,
    sensor_position=sensor_pos,
    method="sample",             # Stochastic sampling based on return probabilities
    return_placement="volume",   # Penetrable spheres have points distributed throughout their interior
    random_seed=42,
)

print(f"Generated {len(point_cloud)} points from {sensor.total_rays} rays.")

# 4. Interactive 3D visualization with vedo
plt = Plotter(N=1, axes=1, bg=(1, 1, 1), interactive=True)
disp = scene.to_vedo(alpha=0.25)
disp.append(point_cloud.to_vedo(color_by="probability", cmap="viridis", point_size=3.5))
plt.show(disp, "10x10m Enclosed Room with Pillars and Low-Density Spheres")


Generated 9281 points from 11520 rays.


## generate training data from multiple viewpoints

In [27]:
sensor_pos = [0.0, 0.0, 1.5]
point_cloud = generate_point_cloud(
    sensor=sensor,
    scene=scene,
    sensor_position=sensor_pos,
    method="sample",             # Stochastic sampling based on return probabilities
    return_placement="volume",   # Penetrable spheres have points distributed throughout their interior
    random_seed=42,
)

cloud_i = np.asarray(np.shape(point_cloud.points))
print(cloud)

[9281    3]
